## 2.1 Forward Conversion to RNS

In the Residue Number System (RNS), an integer number is represented by its residues with respect to a set of pairwise coprime moduli.

Let the modulus set be:

$$
\{m_1, m_2, \dots, m_k\}
$$

The RNS representation of an integer \(x\) is:

$$
RNS(x) = (x \bmod m_1,\ x \bmod m_2,\ \dots,\ x \bmod m_k)
$$

For the given modulus set:

$$
\{2,3,5\}
$$

the dynamic range is:

$$
M = 2 \times 3 \times 5 = 30
$$

Therefore, the unsigned representable range is:

$$
0 \leq x \leq 29
$$

or equivalently:

$$
x \in [0,29]
$$

For signed representation, we use the centered interval:

$$
-\frac{M}{2} \leq x \leq \frac{M}{2}-1
$$

Since \(M = 30\), the signed range is:

$$
-15 \leq x \leq 14
$$

or equivalently:

$$
x \in [-15,14]
$$

Because the moduli \(2\), \(3\), and \(5\) are pairwise coprime, every integer in the valid range has a unique RNS representation.

In [1]:
from math import prod, gcd
import pandas as pd

In [2]:
def check_pairwise_coprime(moduli):
    """
    Check whether all moduli are pairwise coprime.
    """
    for i in range(len(moduli)):
        for j in range(i + 1, len(moduli)):
            if gcd(moduli[i], moduli[j]) != 1:
                return False
    return True


def rns_forward_unsigned(x, moduli):
    """
    Convert an unsigned integer x to RNS representation.
    Valid range: 0 <= x < M
    """
    M = prod(moduli)

    if not check_pairwise_coprime(moduli):
        raise ValueError("The moduli must be pairwise coprime.")

    if not (0 <= x < M):
        raise ValueError(f"x must be in unsigned range [0, {M - 1}].")

    return tuple(x % m for m in moduli)


def signed_range(moduli):
    """
    Return the signed representable range for a modulus set.
    For even M: [-M/2, M/2 - 1]
    For odd M:  [-(M//2), M//2]
    """
    M = prod(moduli)

    if M % 2 == 0:
        return -M // 2, M // 2 - 1
    else:
        return -(M // 2), M // 2


def rns_forward_signed(x, moduli):
    """
    Convert a signed integer x to RNS representation.
    The signed range is centered around zero.
    """
    M = prod(moduli)
    low, high = signed_range(moduli)

    if not check_pairwise_coprime(moduli):
        raise ValueError("The moduli must be pairwise coprime.")

    if not (low <= x <= high):
        raise ValueError(f"x must be in signed range [{low}, {high}].")

    return tuple(x % m for m in moduli)

In [3]:
def verify_uniqueness(values, moduli):
    """
    Verify that no two numbers in the given range have the same RNS representation.
    """
    representations = [tuple(x % m for m in moduli) for x in values]
    return len(representations) == len(set(representations))

In [4]:
moduli = [2, 3, 5]
M = prod(moduli)

print("Moduli:", moduli)
print("Dynamic range M:", M)
print("Pairwise coprime:", check_pairwise_coprime(moduli))

unsigned_values = list(range(0, M))
signed_low, signed_high = signed_range(moduli)
signed_values = list(range(signed_low, signed_high + 1))

print("Unsigned range:", (0, M - 1))
print("Signed range:", (signed_low, signed_high))

Moduli: [2, 3, 5]
Dynamic range M: 30
Pairwise coprime: True
Unsigned range: (0, 29)
Signed range: (-15, 14)


In [5]:
unsigned_table = pd.DataFrame({
    "Unsigned x": unsigned_values,
    "RNS(x)": [rns_forward_unsigned(x, moduli) for x in unsigned_values]
})

unsigned_table

,Unsigned x,RNS(x)
0,0,"(0, 0, 0)"
1,1,"(1, 1, 1)"
2,2,"(0, 2, 2)"
3,3,"(1, 0, 3)"
4,4,"(0, 1, 4)"
5,5,"(1, 2, 0)"
6,6,"(0, 0, 1)"
7,7,"(1, 1, 2)"
8,8,"(0, 2, 3)"
9,9,"(1, 0, 4)"


In [7]:
signed_table = pd.DataFrame({
    "Signed x": signed_values,
    "RNS(x)": [rns_forward_signed(x, moduli) for x in signed_values]
})

signed_table

,Signed x,RNS(x)
0,-15,"(1, 0, 0)"
1,-14,"(0, 1, 1)"
2,-13,"(1, 2, 2)"
3,-12,"(0, 0, 3)"
4,-11,"(1, 1, 4)"
5,-10,"(0, 2, 0)"
6,-9,"(1, 0, 1)"
7,-8,"(0, 1, 2)"
8,-7,"(1, 2, 3)"
9,-6,"(0, 0, 4)"


In [8]:
unsigned_unique = verify_uniqueness(unsigned_values, moduli)
signed_unique = verify_uniqueness(signed_values, moduli)

print("Unsigned RNS representations are unique:", unsigned_unique)
print("Signed RNS representations are unique:", signed_unique)

Unsigned RNS representations are unique: True
Signed RNS representations are unique: True


### Analysis of Results

For the modulus set:

$$
\{2,3,5\}
$$

the dynamic range is:

$$
M = 30
$$

Thus, 30 distinct numbers can be represented.

For the unsigned case, the valid range is:

$$
[0,29]
$$

For the signed case, the valid range is:

$$
[-15,14]
$$

The uniqueness test shows that all numbers in both ranges have unique RNS representations. This happens because the selected moduli \(2\), \(3\), and \(5\) are pairwise coprime.

It is important to note that the same residue vector can correspond to different values depending on whether the interpretation is signed or unsigned.

For example, in unsigned representation:

$$
15 \rightarrow (1,0,0)
$$

while in signed representation:

$$
-15 \rightarrow (1,0,0)
$$

Therefore, uniqueness is guaranteed only within a fixed interpretation range.

## 2.2 Inverse Conversion Using CRT

In this section, we convert numbers from RNS representation back to the integer domain using the Chinese Remainder Theorem (CRT).

Assume the modulus set is:

$$
\{m_1, m_2, \dots, m_k\}
$$

and the RNS representation of a number is:

$$
(r_1, r_2, \dots, r_k)
$$

where each residue is defined as:

$$
r_i = x \bmod m_i
$$

The total dynamic range is:

$$
M = \prod_{i=1}^{k} m_i
$$

For each modulus \(m_i\), define:

$$
M_i = \frac{M}{m_i}
$$

Then define:

$$
y_i = M_i^{-1} \bmod m_i
$$

where \(y_i\) is the modular inverse of \(M_i\) modulo \(m_i\).

Using CRT, the reconstructed unsigned value is:

$$
x = \left( \sum_{i=1}^{k} r_i M_i y_i \right) \bmod M
$$

For the modulus set:

$$
\{2,3,5\}
$$

we have:

$$
M = 2 \times 3 \times 5 = 30
$$

Therefore, the unsigned output range is:

$$
x \in [0,29]
$$

For signed reconstruction, we first reconstruct the unsigned value \(u\), then map it to the signed range:

$$
[-15,14]
$$

The signed conversion rule is:

$$
x =
\begin{cases}
u, & u < \frac{M}{2} \\
u - M, & u \geq \frac{M}{2}
\end{cases}
$$

Since \(M = 30\), values from \(0\) to \(14\) remain unchanged, while values from \(15\) to \(29\) are interpreted as negative numbers from \(-15\) to \(-1\).

In [9]:
def extended_gcd(a, b):
    """
    Extended Euclidean algorithm.
    Returns gcd(a,b), x, y such that ax + by = gcd(a,b).
    """
    if b == 0:
        return a, 1, 0

    g, x1, y1 = extended_gcd(b, a % b)
    x = y1
    y = x1 - (a // b) * y1

    return g, x, y


def mod_inverse(a, m):
    """
    Compute modular inverse of a modulo m.
    """
    g, x, _ = extended_gcd(a, m)

    if g != 1:
        raise ValueError(f"{a} has no modular inverse modulo {m}.")

    return x % m

In [10]:
def crt_inverse_unsigned(rns_value, moduli):
    """
    Convert an RNS representation back to an unsigned integer using CRT.
    Output range: [0, M-1]
    """
    if not check_pairwise_coprime(moduli):
        raise ValueError("The moduli must be pairwise coprime.")

    if len(rns_value) != len(moduli):
        raise ValueError("Length of RNS value must match number of moduli.")

    M = prod(moduli)
    total = 0

    for residue, m_i in zip(rns_value, moduli):
        if not (0 <= residue < m_i):
            raise ValueError(f"Residue {residue} is invalid for modulus {m_i}.")

        M_i = M // m_i
        y_i = mod_inverse(M_i, m_i)

        total += residue * M_i * y_i

    return total % M


def crt_inverse_signed(rns_value, moduli):
    """
    Convert an RNS representation back to a signed integer using CRT.
    Signed range is centered around zero.
    """
    M = prod(moduli)
    unsigned_value = crt_inverse_unsigned(rns_value, moduli)

    if M % 2 == 0:
        if unsigned_value >= M // 2:
            return unsigned_value - M
        else:
            return unsigned_value
    else:
        if unsigned_value > M // 2:
            return unsigned_value - M
        else:
            return unsigned_value

In [11]:
moduli = [2, 3, 5]
M = prod(moduli)

crt_terms = []

for m_i in moduli:
    M_i = M // m_i
    y_i = mod_inverse(M_i, m_i)

    crt_terms.append({
        "m_i": m_i,
        "M_i = M / m_i": M_i,
        "y_i = inverse(M_i mod m_i)": y_i
    })

crt_terms_table = pd.DataFrame(crt_terms)
crt_terms_table

,m_i,M_i = M / m_i,y_i = inverse(M_i mod m_i)
0,2,15,1
1,3,10,1
2,5,6,1


For the modulus set \(\{2,3,5\}\), all CRT modular inverses are equal to 1:

\[
M_1 = 15,\quad 15^{-1} \bmod 2 = 1
\]

\[
M_2 = 10,\quad 10^{-1} \bmod 3 = 1
\]

\[
M_3 = 6,\quad 6^{-1} \bmod 5 = 1
\]

Therefore, for this specific modulus set, the CRT reconstruction formula becomes:

\[
x = \langle 15r_1 + 10r_2 + 6r_3 \rangle_{30}
\]

In [12]:
unsigned_values = list(range(0, M))

unsigned_crt_table = pd.DataFrame({
    "Unsigned x": unsigned_values,
    "RNS(x)": [rns_forward_unsigned(x, moduli) for x in unsigned_values],
})

unsigned_crt_table["CRT inverse unsigned"] = unsigned_crt_table["RNS(x)"].apply(
    lambda rns: crt_inverse_unsigned(rns, moduli)
)

unsigned_crt_table["Correct"] = (
    unsigned_crt_table["Unsigned x"] == unsigned_crt_table["CRT inverse unsigned"]
)

unsigned_crt_table

,Unsigned x,RNS(x),CRT inverse unsigned,Correct
0,0,"(0, 0, 0)",0,True
1,1,"(1, 1, 1)",1,True
2,2,"(0, 2, 2)",2,True
3,3,"(1, 0, 3)",3,True
4,4,"(0, 1, 4)",4,True
5,5,"(1, 2, 0)",5,True
6,6,"(0, 0, 1)",6,True
7,7,"(1, 1, 2)",7,True
8,8,"(0, 2, 3)",8,True
9,9,"(1, 0, 4)",9,True


In [13]:
signed_low, signed_high = signed_range(moduli)
signed_values = list(range(signed_low, signed_high + 1))

signed_crt_table = pd.DataFrame({
    "Signed x": signed_values,
    "RNS(x)": [rns_forward_signed(x, moduli) for x in signed_values],
})

signed_crt_table["CRT inverse signed"] = signed_crt_table["RNS(x)"].apply(
    lambda rns: crt_inverse_signed(rns, moduli)
)

signed_crt_table["Correct"] = (
    signed_crt_table["Signed x"] == signed_crt_table["CRT inverse signed"]
)

signed_crt_table

,Signed x,RNS(x),CRT inverse signed,Correct
0,-15,"(1, 0, 0)",-15,True
1,-14,"(0, 1, 1)",-14,True
2,-13,"(1, 2, 2)",-13,True
3,-12,"(0, 0, 3)",-12,True
4,-11,"(1, 1, 4)",-11,True
5,-10,"(0, 2, 0)",-10,True
6,-9,"(1, 0, 1)",-9,True
7,-8,"(0, 1, 2)",-8,True
8,-7,"(1, 2, 3)",-7,True
9,-6,"(0, 0, 4)",-6,True


In [14]:
print("Unsigned CRT inverse is correct for all values:",
      unsigned_crt_table["Correct"].all())

print("Signed CRT inverse is correct for all values:",
      signed_crt_table["Correct"].all())

Unsigned CRT inverse is correct for all values: True
Signed CRT inverse is correct for all values: True


### CRT Coefficients for the Modulus Set

For the modulus set:

$$
\{2,3,5\}
$$

the dynamic range is:

$$
M = 30
$$

For \(m_1 = 2\):

$$
M_1 = \frac{30}{2} = 15
$$

$$
y_1 = 15^{-1} \bmod 2 = 1
$$

For \(m_2 = 3\):

$$
M_2 = \frac{30}{3} = 10
$$

$$
y_2 = 10^{-1} \bmod 3 = 1
$$

For \(m_3 = 5\):

$$
M_3 = \frac{30}{5} = 6
$$

$$
y_3 = 6^{-1} \bmod 5 = 1
$$

Therefore, for this specific modulus set, the CRT reconstruction formula becomes:

$$
x = (15r_1 + 10r_2 + 6r_3) \bmod 30
$$

### Analysis of Results

The CRT inverse conversion correctly reconstructs all unsigned values in the range:

$$
[0,29]
$$

It also correctly reconstructs all signed values in the range:

$$
[-15,14]
$$

The important point is that the same RNS vector may represent different values depending on the interpretation mode.

For example, the residue vector:

$$
(1,0,0)
$$

corresponds to:

$$
15
$$

in unsigned interpretation, but corresponds to:

$$
-15
$$

in signed interpretation.

Therefore, the RNS representation itself does not explicitly store the sign. The sign is determined by the selected interpretation range during inverse conversion.

## 2.3 Inverse Conversion Using MRC

In this section, we convert numbers from RNS representation back to the integer domain using Mixed Radix Conversion (MRC).

In MRC, the integer \(x\) is represented in mixed-radix form as:

$$
x = k_1 + k_2 m_1 + k_3 m_1m_2 + \cdots + k_n m_1m_2\cdots m_{n-1}
$$

where \(k_1, k_2, \dots, k_n\) are the mixed-radix coefficients.

For the modulus set:

$$
\{m_1, m_2, \dots, m_n\}
$$

the RNS representation is:

$$
(r_1, r_2, \dots, r_n)
$$

where:

$$
r_i = x \bmod m_i
$$

Garner's algorithm computes the mixed-radix coefficients sequentially. The first coefficient is:

$$
k_1 = r_1
$$

The next coefficients are computed using modular inverses. For example:

$$
k_2 = (r_2 - k_1)m_1^{-1} \bmod m_2
$$

and:

$$
k_3 = \left((r_3 - k_1)m_1^{-1} - k_2\right)m_2^{-1} \bmod m_3
$$

After computing the coefficients, the integer value is reconstructed using the mixed-radix expansion.

For the modulus set:

$$
\{2,3,5\}
$$

the dynamic range is:

$$
M = 2 \times 3 \times 5 = 30
$$

Thus, the unsigned range is:

$$
[0,29]
$$

and the signed range is:

$$
[-15,14]
$$

In [15]:
def mrc_coefficients(rns_value, moduli):
    """
    Compute mixed-radix coefficients from an RNS representation
    using Garner's algorithm.

    The result depends on the order of moduli, but the reconstructed
    integer is unique within the dynamic range.
    """
    if not check_pairwise_coprime(moduli):
        raise ValueError("The moduli must be pairwise coprime.")

    if len(rns_value) != len(moduli):
        raise ValueError("Length of RNS value must match number of moduli.")

    for residue, modulus in zip(rns_value, moduli):
        if not (0 <= residue < modulus):
            raise ValueError(f"Residue {residue} is invalid for modulus {modulus}.")

    n = len(moduli)
    coeffs = [0] * n

    for i in range(n):
        value = rns_value[i]

        for j in range(i):
            inv = mod_inverse(moduli[j], moduli[i])
            value = ((value - coeffs[j]) * inv) % moduli[i]

        coeffs[i] = value

    return tuple(coeffs)


def mixed_radix_to_integer(coeffs, moduli):
    """
    Reconstruct the integer value from mixed-radix coefficients.
    """
    x = 0
    multiplier = 1

    for i, coeff in enumerate(coeffs):
        x += coeff * multiplier

        if i < len(moduli):
            multiplier *= moduli[i]

    return x

In [16]:
def mrc_inverse_unsigned(rns_value, moduli):
    """
    Convert an RNS representation back to an unsigned integer using MRC.
    Output range: [0, M-1]
    """
    M = prod(moduli)

    coeffs = mrc_coefficients(rns_value, moduli)
    x = mixed_radix_to_integer(coeffs, moduli)

    return x % M


def mrc_inverse_signed(rns_value, moduli):
    """
    Convert an RNS representation back to a signed integer using MRC.
    The signed range is centered around zero.
    """
    M = prod(moduli)

    unsigned_value = mrc_inverse_unsigned(rns_value, moduli)

    if M % 2 == 0:
        if unsigned_value >= M // 2:
            return unsigned_value - M
        else:
            return unsigned_value
    else:
        if unsigned_value > M // 2:
            return unsigned_value - M
        else:
            return unsigned_value

### MRC Formula for the Modulus Set

For the modulus set:

$$
\{2,3,5\}
$$

the mixed-radix representation is:

$$
x = k_1 + k_2(2) + k_3(2)(3)
$$

Therefore:

$$
x = k_1 + 2k_2 + 6k_3
$$

where:

$$
0 \leq k_1 < 2
$$

$$
0 \leq k_2 < 3
$$

$$
0 \leq k_3 < 5
$$

The coefficients \(k_1\), \(k_2\), and \(k_3\) are computed from the RNS residues using Garner's algorithm.

In [17]:
moduli = [2, 3, 5]
M = prod(moduli)

unsigned_values = list(range(0, M))

unsigned_mrc_table = pd.DataFrame({
    "Unsigned x": unsigned_values,
    "RNS(x)": [rns_forward_unsigned(x, moduli) for x in unsigned_values]
})

unsigned_mrc_table["MRC coefficients"] = unsigned_mrc_table["RNS(x)"].apply(
    lambda rns: mrc_coefficients(rns, moduli)
)

unsigned_mrc_table["MRC inverse unsigned"] = unsigned_mrc_table["RNS(x)"].apply(
    lambda rns: mrc_inverse_unsigned(rns, moduli)
)

unsigned_mrc_table["Correct"] = (
    unsigned_mrc_table["Unsigned x"] == unsigned_mrc_table["MRC inverse unsigned"]
)

unsigned_mrc_table

,Unsigned x,RNS(x),MRC coefficients,MRC inverse unsigned,Correct
0,0,"(0, 0, 0)","(0, 0, 0)",0,True
1,1,"(1, 1, 1)","(1, 0, 0)",1,True
2,2,"(0, 2, 2)","(0, 1, 0)",2,True
3,3,"(1, 0, 3)","(1, 1, 0)",3,True
4,4,"(0, 1, 4)","(0, 2, 0)",4,True
5,5,"(1, 2, 0)","(1, 2, 0)",5,True
6,6,"(0, 0, 1)","(0, 0, 1)",6,True
7,7,"(1, 1, 2)","(1, 0, 1)",7,True
8,8,"(0, 2, 3)","(0, 1, 1)",8,True
9,9,"(1, 0, 4)","(1, 1, 1)",9,True


In [18]:
signed_low, signed_high = signed_range(moduli)
signed_values = list(range(signed_low, signed_high + 1))

signed_mrc_table = pd.DataFrame({
    "Signed x": signed_values,
    "RNS(x)": [rns_forward_signed(x, moduli) for x in signed_values]
})

signed_mrc_table["MRC coefficients"] = signed_mrc_table["RNS(x)"].apply(
    lambda rns: mrc_coefficients(rns, moduli)
)

signed_mrc_table["MRC inverse signed"] = signed_mrc_table["RNS(x)"].apply(
    lambda rns: mrc_inverse_signed(rns, moduli)
)

signed_mrc_table["Correct"] = (
    signed_mrc_table["Signed x"] == signed_mrc_table["MRC inverse signed"]
)

signed_mrc_table

,Signed x,RNS(x),MRC coefficients,MRC inverse signed,Correct
0,-15,"(1, 0, 0)","(1, 1, 2)",-15,True
1,-14,"(0, 1, 1)","(0, 2, 2)",-14,True
2,-13,"(1, 2, 2)","(1, 2, 2)",-13,True
3,-12,"(0, 0, 3)","(0, 0, 3)",-12,True
4,-11,"(1, 1, 4)","(1, 0, 3)",-11,True
5,-10,"(0, 2, 0)","(0, 1, 3)",-10,True
6,-9,"(1, 0, 1)","(1, 1, 3)",-9,True
7,-8,"(0, 1, 2)","(0, 2, 3)",-8,True
8,-7,"(1, 2, 3)","(1, 2, 3)",-7,True
9,-6,"(0, 0, 4)","(0, 0, 4)",-6,True


In [19]:
comparison_table = pd.DataFrame({
    "Unsigned x": unsigned_values,
    "RNS(x)": [rns_forward_unsigned(x, moduli) for x in unsigned_values]
})

comparison_table["CRT inverse"] = comparison_table["RNS(x)"].apply(
    lambda rns: crt_inverse_unsigned(rns, moduli)
)

comparison_table["MRC inverse"] = comparison_table["RNS(x)"].apply(
    lambda rns: mrc_inverse_unsigned(rns, moduli)
)

comparison_table["CRT equals MRC"] = (
    comparison_table["CRT inverse"] == comparison_table["MRC inverse"]
)

comparison_table

,Unsigned x,RNS(x),CRT inverse,MRC inverse,CRT equals MRC
0,0,"(0, 0, 0)",0,0,True
1,1,"(1, 1, 1)",1,1,True
2,2,"(0, 2, 2)",2,2,True
3,3,"(1, 0, 3)",3,3,True
4,4,"(0, 1, 4)",4,4,True
5,5,"(1, 2, 0)",5,5,True
6,6,"(0, 0, 1)",6,6,True
7,7,"(1, 1, 2)",7,7,True
8,8,"(0, 2, 3)",8,8,True
9,9,"(1, 0, 4)",9,9,True


### Analysis of Results

The MRC inverse conversion correctly reconstructs all unsigned values in the range:

$$
[0,29]
$$

It also correctly reconstructs all signed values in the range:

$$
[-15,14]
$$

The comparison between CRT and MRC shows that both methods reconstruct the same integer value for every RNS vector in the valid dynamic range.

The difference between the two methods is in the reconstruction procedure. CRT reconstructs the integer using a direct summation formula modulo \(M\), while MRC reconstructs the integer by first computing mixed-radix coefficients and then evaluating the mixed-radix expansion.

For the modulus set:

$$
\{2,3,5\}
$$

the mixed-radix form is:

$$
x = k_1 + 2k_2 + 6k_3
$$

The result confirms that Garner's algorithm successfully computes the coefficients \(k_1\), \(k_2\), and \(k_3\), and that MRC can be used as an alternative inverse-conversion method for RNS.

## 2.4 Arithmetic Operations in RNS

In this section, we implement arithmetic operations directly in the RNS domain.

The required operations are:

1. Addition
2. Subtraction
3. Multiplication

Let two numbers \(A\) and \(B\) be represented in RNS as:

$$
A = (a_1, a_2, \dots, a_n)
$$

and:

$$
B = (b_1, b_2, \dots, b_n)
$$

with respect to the modulus set:

$$
\{m_1, m_2, \dots, m_n\}
$$

Then RNS addition is performed independently in each residue channel:

$$
A + B =
((a_1+b_1)\bmod m_1,\ (a_2+b_2)\bmod m_2,\ \dots,\ (a_n+b_n)\bmod m_n)
$$

RNS subtraction is:

$$
A - B =
((a_1-b_1)\bmod m_1,\ (a_2-b_2)\bmod m_2,\ \dots,\ (a_n-b_n)\bmod m_n)
$$

RNS multiplication is:

$$
A \times B =
((a_1b_1)\bmod m_1,\ (a_2b_2)\bmod m_2,\ \dots,\ (a_nb_n)\bmod m_n)
$$

For this section, the modulus set is:

$$
\{3,4,5\}
$$

The dynamic range is:

$$
M = 3 \times 4 \times 5 = 60
$$

Therefore, the unsigned range is:

$$
[0,59]
$$

For signed representation, we use the centered range:

$$
[-30,29]
$$

To avoid overflow, the generated test pairs are selected such that the true arithmetic result remains inside the signed range:

$$
[-30,29]
$$

In [20]:
def rns_add(a_rns, b_rns, moduli):
    """
    Add two RNS numbers residue-wise.
    """
    if len(a_rns) != len(moduli) or len(b_rns) != len(moduli):
        raise ValueError("RNS vector length must match the number of moduli.")

    return tuple((a_i + b_i) % m_i for a_i, b_i, m_i in zip(a_rns, b_rns, moduli))


def rns_sub(a_rns, b_rns, moduli):
    """
    Subtract two RNS numbers residue-wise.
    """
    if len(a_rns) != len(moduli) or len(b_rns) != len(moduli):
        raise ValueError("RNS vector length must match the number of moduli.")

    return tuple((a_i - b_i) % m_i for a_i, b_i, m_i in zip(a_rns, b_rns, moduli))


def rns_mul(a_rns, b_rns, moduli):
    """
    Multiply two RNS numbers residue-wise.
    """
    if len(a_rns) != len(moduli) or len(b_rns) != len(moduli):
        raise ValueError("RNS vector length must match the number of moduli.")

    return tuple((a_i * b_i) % m_i for a_i, b_i, m_i in zip(a_rns, b_rns, moduli))

In [21]:

def apply_operation(x, y, operation):
    """
    Apply the selected operation in the integer domain.
    """
    if operation == "add":
        return x + y
    elif operation == "sub":
        return x - y
    elif operation == "mul":
        return x * y
    else:
        raise ValueError("Operation must be 'add', 'sub', or 'mul'.")


def apply_rns_operation(a_rns, b_rns, moduli, operation):
    """
    Apply the selected operation in the RNS domain.
    """
    if operation == "add":
        return rns_add(a_rns, b_rns, moduli)
    elif operation == "sub":
        return rns_sub(a_rns, b_rns, moduli)
    elif operation == "mul":
        return rns_mul(a_rns, b_rns, moduli)
    else:
        raise ValueError("Operation must be 'add', 'sub', or 'mul'.")

In [26]:
import random
def generate_safe_pairs(moduli, operation, count=5, seed=0):
    """
    Generate random signed integer pairs for which the result does not overflow.
    """
    random.seed(seed)

    low, high = signed_range(moduli)
    valid_values = list(range(low, high + 1))

    pairs = []

    while len(pairs) < count:
        x = random.choice(valid_values)
        y = random.choice(valid_values)

        result = apply_operation(x, y, operation)

        if low <= result <= high:
            pairs.append((x, y))

    return pairs

In [27]:
def build_operation_table(moduli, operation, count=5, seed=0):
    """
    Build a verification table for one RNS arithmetic operation.
    """
    pairs = generate_safe_pairs(moduli, operation, count=count, seed=seed)

    rows = []

    for x, y in pairs:
        x_rns = rns_forward_signed(x, moduli)
        y_rns = rns_forward_signed(y, moduli)

        rns_result = apply_rns_operation(x_rns, y_rns, moduli, operation)
        inverse_result = crt_inverse_signed(rns_result, moduli)

        expected_result = apply_operation(x, y, operation)

        rows.append({
            "x": x,
            "y": y,
            "RNS(x)": x_rns,
            "RNS(y)": y_rns,
            "RNS result": rns_result,
            "Inverse result": inverse_result,
            "Expected result": expected_result,
            "Correct": inverse_result == expected_result
        })

    return pd.DataFrame(rows)

In [28]:
moduli_24 = [3, 4, 5]
M_24 = prod(moduli_24)

low_24, high_24 = signed_range(moduli_24)

print("Moduli:", moduli_24)
print("Dynamic range M:", M_24)
print("Signed range:", (low_24, high_24))
print("Pairwise coprime:", check_pairwise_coprime(moduli_24))

Moduli: [3, 4, 5]
Dynamic range M: 60
Signed range: (-30, 29)
Pairwise coprime: True


### Addition in RNS

For addition, each residue channel is added independently:

$$
RNS(x+y) =
((x_1+y_1)\bmod m_1,\ (x_2+y_2)\bmod m_2,\ (x_3+y_3)\bmod m_3)
$$

The following table verifies addition using five random input pairs whose true results are inside the signed range:

$$
[-30,29]
$$

In [31]:
addition_table = build_operation_table(
    moduli=moduli_24,
    operation="add",
    count=5,
    seed=1
)

addition_table

,x,y,RNS(x),RNS(y),RNS result,Inverse result,Expected result,Correct
0,-22,6,"(2, 2, 3)","(0, 2, 1)","(2, 0, 4)",-16,-16,True
1,18,-26,"(0, 2, 3)","(1, 2, 4)","(1, 0, 2)",-8,-8,True
2,1,18,"(1, 1, 1)","(0, 2, 3)","(1, 3, 4)",19,19,True
3,-2,0,"(1, 2, 3)","(0, 0, 0)","(1, 2, 3)",-2,-2,True
4,11,-6,"(2, 3, 1)","(0, 2, 4)","(2, 1, 0)",5,5,True


In [32]:
print("Addition is correct for all test pairs:",
      addition_table["Correct"].all())

Addition is correct for all test pairs: True


### Subtraction in RNS

For subtraction, each residue channel is subtracted independently:

$$
RNS(x-y) =
((x_1-y_1)\bmod m_1,\ (x_2-y_2)\bmod m_2,\ (x_3-y_3)\bmod m_3)
$$

The following table verifies subtraction using five random input pairs whose true results are inside the signed range:

$$
[-30,29]
$$

In [33]:
subtraction_table = build_operation_table(
    moduli=moduli_24,
    operation="sub",
    count=5,
    seed=2
)

subtraction_table

,x,y,RNS(x),RNS(y),RNS result,Inverse result,Expected result,Correct
0,25,24,"(1, 1, 0)","(0, 0, 4)","(1, 1, 1)",1,1,True
1,-27,-25,"(0, 1, 3)","(2, 3, 0)","(1, 2, 3)",-2,-2,True
2,-25,-7,"(2, 3, 0)","(2, 1, 3)","(0, 2, 2)",-18,-18,True
3,17,21,"(2, 1, 2)","(0, 1, 1)","(2, 0, 1)",-4,-4,True
4,12,24,"(0, 0, 2)","(0, 0, 4)","(0, 0, 3)",-12,-12,True


### Multiplication in RNS

For multiplication, each residue channel is multiplied independently:

$$
RNS(x \times y) =
((x_1y_1)\bmod m_1,\ (x_2y_2)\bmod m_2,\ (x_3y_3)\bmod m_3)
$$

The following table verifies multiplication using five random input pairs whose true results are inside the signed range:

$$
[-30,29]
$$

In [35]:
multiplication_table = build_operation_table(
    moduli=moduli_24,
    operation="mul",
    count=5,
    seed=4
)

multiplication_table

,x,y,RNS(x),RNS(y),RNS result,Inverse result,Expected result,Correct
0,-5,0,"(1, 3, 0)","(0, 0, 0)","(0, 0, 0)",0,0,True
1,-5,5,"(1, 3, 0)","(2, 1, 0)","(2, 3, 0)",-25,-25,True
2,3,4,"(0, 3, 3)","(1, 0, 4)","(0, 0, 2)",12,12,True
3,-6,2,"(0, 2, 4)","(2, 2, 2)","(0, 0, 3)",-12,-12,True
4,-15,0,"(0, 1, 0)","(0, 0, 0)","(0, 0, 0)",0,0,True


In [36]:
print("Multiplication is correct for all test pairs:",
      multiplication_table["Correct"].all())

Multiplication is correct for all test pairs: True


### Analysis of Results

The RNS arithmetic operations correctly produce the expected results for all generated test pairs.

For the modulus set:

$$
\{3,4,5\}
$$

the dynamic range is:

$$
M = 60
$$

and the signed range is:

$$
[-30,29]
$$

For each operation, two signed integers were first converted to RNS. Then the operation was performed directly in the residue domain. Finally, the RNS result was converted back to the signed integer domain using CRT.

The results show that:

$$
RNS(x+y)
$$

matches the inverse-converted result of residue-wise addition,

$$
RNS(x-y)
$$

matches the inverse-converted result of residue-wise subtraction, and

$$
RNS(x \times y)
$$

matches the inverse-converted result of residue-wise multiplication.

This confirms that addition, subtraction, and multiplication can be performed independently in each residue channel.

The main advantage of this property is that arithmetic in RNS does not require carry propagation between residue channels. Therefore, these operations can be implemented in parallel hardware channels, which can improve speed and reduce arithmetic complexity.

However, these results are valid only when overflow does not occur. If the true result goes outside the selected signed range:

$$
[-30,29]
$$

then the RNS result wraps around modulo:

$$
M = 60
$$

and the inverse-converted value will no longer match the ordinary integer-domain result.

## 2.5 Application of RNS in a Small CNN Model

In this section, we implement the computation of a small CNN layer using the Residue Number System.

The modulus set is:

$$
\{7,8,9\}
$$

The dynamic range is:

$$
M = 7 \times 8 \times 9 = 504
$$

The input and weight values are randomly selected from the range:

$$
[-256,255]
$$

The CNN layer has the following structure:

- Input shape:

$$
3 \times 8 \times 8
$$

- Kernel shape:

$$
3 \times 5 \times 5
$$

- Output shape:

$$
1 \times 4 \times 4
$$

This is a valid convolution because:

$$
8 - 5 + 1 = 4
$$

Therefore, the output spatial size is:

$$
4 \times 4
$$

The convolution output at position \((i,j)\) is computed as:

$$
Y[i,j] =
\sum_{c=0}^{2}
\sum_{u=0}^{4}
\sum_{v=0}^{4}
X[c,i+u,j+v] \times W[c,u,v]
$$

In the RNS domain, multiplication and addition are performed independently in each residue channel.

Since RNS arithmetic with moduli \(\{7,8,9\}\) represents values modulo:

$$
M = 504
$$

the real-domain convolution result must be converted to its residue modulo \(504\) before comparison:

$$
Y_{\text{real, mod}} = Y_{\text{real}} \bmod 504
$$

This is necessary because RNS arithmetic wraps around modulo \(M\).

In [37]:
import numpy as np
def rns_forward_general(x, moduli):
    """
    Convert any integer x to its RNS representation.
    
    This version does not restrict x to the signed or unsigned range.
    It simply computes residues modulo each modulus.
    """
    return tuple(int(x) % m for m in moduli)


def rns_to_unsigned_crt(rns_value, moduli):
    """
    Convert RNS value to unsigned integer in [0, M-1] using CRT.
    """
    return crt_inverse_unsigned(rns_value, moduli)

In [38]:
def rns_mac(a_rns, b_rns, acc_rns, moduli):
    """
    Perform one multiply-accumulate operation in RNS:
    
    acc = acc + a * b
    """
    product_rns = rns_mul(a_rns, b_rns, moduli)
    acc_rns = rns_add(acc_rns, product_rns, moduli)
    return acc_rns

In [39]:
def conv2d_real(input_tensor, kernel):
    """
    Perform valid convolution in the real/integer domain.
    
    input_tensor shape: (C, H, W)
    kernel shape:       (C, KH, KW)
    output shape:       (H-KH+1, W-KW+1)
    """
    C, H, W = input_tensor.shape
    Ck, KH, KW = kernel.shape

    if C != Ck:
        raise ValueError("Input and kernel must have the same number of channels.")

    OH = H - KH + 1
    OW = W - KW + 1

    output = np.zeros((OH, OW), dtype=int)

    for i in range(OH):
        for j in range(OW):
            total = 0

            for c in range(C):
                for u in range(KH):
                    for v in range(KW):
                        total += int(input_tensor[c, i + u, j + v]) * int(kernel[c, u, v])

            output[i, j] = total

    return output

In [40]:
def conv2d_rns(input_tensor, kernel, moduli):
    """
    Perform valid convolution in the RNS domain.
    
    The result is returned in two forms:
    1. RNS output, where each output element is an RNS tuple.
    2. Unsigned reconstructed output in [0, M-1].
    """
    C, H, W = input_tensor.shape
    Ck, KH, KW = kernel.shape

    if C != Ck:
        raise ValueError("Input and kernel must have the same number of channels.")

    OH = H - KH + 1
    OW = W - KW + 1

    zero_rns = tuple(0 for _ in moduli)

    output_rns = [[None for _ in range(OW)] for _ in range(OH)]
    output_unsigned = np.zeros((OH, OW), dtype=int)

    for i in range(OH):
        for j in range(OW):
            acc_rns = zero_rns

            for c in range(C):
                for u in range(KH):
                    for v in range(KW):
                        x_rns = rns_forward_general(input_tensor[c, i + u, j + v], moduli)
                        w_rns = rns_forward_general(kernel[c, u, v], moduli)

                        acc_rns = rns_mac(x_rns, w_rns, acc_rns, moduli)

            output_rns[i][j] = acc_rns
            output_unsigned[i, j] = rns_to_unsigned_crt(acc_rns, moduli)

    return output_rns, output_unsigned

In [41]:
moduli_25 = [7, 8, 9]
M_25 = prod(moduli_25)

print("Moduli:", moduli_25)
print("Dynamic range M:", M_25)
print("Pairwise coprime:", check_pairwise_coprime(moduli_25))

Moduli: [7, 8, 9]
Dynamic range M: 504
Pairwise coprime: True


In [42]:
np.random.seed(0)

input_tensor = np.random.randint(
    low=-256,
    high=256,
    size=(3, 8, 8)
)

kernel = np.random.randint(
    low=-256,
    high=256,
    size=(3, 5, 5)
)

print("Input shape:", input_tensor.shape)
print("Kernel shape:", kernel.shape)

Input shape: (3, 8, 8)
Kernel shape: (3, 5, 5)


In [43]:
real_output = conv2d_real(input_tensor, kernel)

real_output_mod = real_output % M_25

rns_output, rns_output_unsigned = conv2d_rns(
    input_tensor=input_tensor,
    kernel=kernel,
    moduli=moduli_25
)

print("Real output shape:", real_output.shape)
print("RNS output shape:", rns_output_unsigned.shape)

Real output shape: (4, 4)
RNS output shape: (4, 4)


In [44]:
print("Real-domain convolution output:")
print(real_output)

Real-domain convolution output:
[[ 199866  263844   52670 -110371]
 [ 368098 -184441  253797   29296]
 [ 340693 -295834   82750  314367]
 [ 134949 -137598 -194116  339648]]


In [45]:
print("RNS-domain output converted back using CRT:")
print(rns_output_unsigned)

RNS-domain output converted back using CRT:
[[282 252 254   5]
 [178  23 285  64]
 [493  14  94 375]
 [381 498 428 456]]


In [46]:
comparison = real_output_mod == rns_output_unsigned

print("Element-wise comparison:")
print(comparison)

print("All outputs match:", comparison.all())

Element-wise comparison:
[[ True  True  True  True]
 [ True  True  True  True]
 [ True  True  True  True]
 [ True  True  True  True]]
All outputs match: True


In [47]:
rows = []

for i in range(real_output.shape[0]):
    for j in range(real_output.shape[1]):
        rows.append({
            "Output position": f"({i},{j})",
            "Real output": real_output[i, j],
            "Real output mod 504": real_output_mod[i, j],
            "RNS output": rns_output[i][j],
            "CRT inverse of RNS output": rns_output_unsigned[i, j],
            "Match": comparison[i, j]
        })

cnn_comparison_table = pd.DataFrame(rows)
cnn_comparison_table

,Output position,Real output,Real output mod 504,RNS output,CRT inverse of RNS output,Match
0,"(0,0)",199866,282,"(2, 2, 3)",282,True
1,"(0,1)",263844,252,"(0, 4, 0)",252,True
2,"(0,2)",52670,254,"(2, 6, 2)",254,True
3,"(0,3)",-110371,5,"(5, 5, 5)",5,True
4,"(1,0)",368098,178,"(3, 2, 7)",178,True
5,"(1,1)",-184441,23,"(2, 7, 5)",23,True
6,"(1,2)",253797,285,"(5, 5, 6)",285,True
7,"(1,3)",29296,64,"(1, 0, 1)",64,True
8,"(2,0)",340693,493,"(3, 5, 7)",493,True
9,"(2,1)",-295834,14,"(0, 6, 5)",14,True


### ReLU Output in the Real Domain



In the real domain, ReLU is defined as:

$$
ReLU(x) = \max(0,x)
$$

Therefore, after computing the convolution output, the ReLU output is:

$$
Y_{\text{ReLU}} = \max(0, Y)
$$

However, ReLU is not a simple residue-wise operation in RNS because it requires sign comparison. RNS arithmetic naturally supports modular addition, subtraction, and multiplication, but comparison and sign detection are non-trivial.

Therefore, the direct RNS computation above verifies the convolution arithmetic modulo \(M=504\). The ReLU output can be computed in the real domain, or after converting the RNS result back to an integer representation if the correct signed interpretation is available.

In [48]:
real_relu_output = np.maximum(real_output, 0)
real_relu_output_mod = real_relu_output % M_25

print("Real-domain ReLU output:")
print(real_relu_output)

print("Real-domain ReLU output modulo 504:")
print(real_relu_output_mod)

Real-domain ReLU output:
[[199866 263844  52670      0]
 [368098      0 253797  29296]
 [340693      0  82750 314367]
 [134949      0      0 339648]]
Real-domain ReLU output modulo 504:
[[282 252 254   0]
 [178   0 285  64]
 [493   0  94 375]
 [381   0   0 456]]


### Analysis of Results

The RNS convolution output matches the real-domain convolution output after reducing the real-domain output modulo:

$$
M = 504
$$

This confirms that the RNS-domain multiply-accumulate operations are mathematically equivalent to integer-domain convolution modulo \(M\).

For every output position:

$$
Y_{\text{RNS}} = Y_{\text{real}} \bmod 504
$$

The reason is that each multiplication and addition in RNS is performed independently under the moduli:

$$
7,\ 8,\ 9
$$

and the combined result represents the final value modulo:

$$
504
$$

The advantage of the RNS implementation is that each residue channel can be computed independently. Therefore, the convolution multiply-accumulate operations can be parallelized across residue channels.

The limitation is that values outside the dynamic range wrap around modulo \(M\). In this example, convolution outputs can be much larger than \(504\), so direct equality with the real-domain output is not expected. The correct comparison is between the RNS output and the real-domain output modulo \(504\).

Also, ReLU is not directly residue-wise because it requires comparison with zero. Therefore, applying ReLU inside the RNS domain requires additional sign-detection or conversion logic.